# Fine-tuning Phi-3-mini-4k-instruct com QLoRA no FinQA

**Objetivo:** Adaptar um LLM pequeno para o domínio de finanças corporativas usando QLoRA.

**Stack:** `transformers` · `peft` · `trl` · `bitsandbytes` · `datasets`

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

---

**Antes de rodar:** adicione seu `HF_TOKEN` nos Secrets do Colab (ícone 🔑 na barra lateral)

In [2]:
import torch

assert torch.cuda.is_available(), 'GPU nao detectada! Va em Runtime -> Change runtime type -> T4 GPU'

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024**3
print(f'GPU: {gpu.name}')
print(f'VRAM: {vram_gb:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

GPU: Tesla T4
VRAM: 14.6 GB
CUDA: 12.8


## 1. Instalar dependencias

In [3]:
%%capture
!pip install -q -U \
    transformers \
    peft \
    trl \
    bitsandbytes \
    datasets \
    accelerate \
    evaluate \
    rouge_score

print('Dependencias instaladas')

In [4]:
import transformers, peft, trl, bitsandbytes, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"datasets: {datasets.__version__}")

transformers: 5.4.0
peft: 0.18.1
trl: 0.29.1
bitsandbytes: 0.49.2
datasets: 4.8.4


## 2. Configurar HF Token

In [5]:
import os

# Se estiver no Colab web, usa Secrets. Se estiver no VS Code, pede via input.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = input('Cole seu HF_TOKEN aqui: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN
assert HF_TOKEN, 'HF_TOKEN nao encontrado!'
print('HF_TOKEN carregado com sucesso')

HF_TOKEN carregado com sucesso


## 3. Carregar e preparar o FinQA

In [6]:
import random
from datasets import load_dataset, Dataset

SYSTEM_PROMPT = (
    'You are an expert assistant in corporate finance. '
    'Answer questions about balance sheets, income statements, '
    'cash flow and accounting terms clearly and accurately.'
)

def format_phi3(example):
    question = example.get('question', '').strip()
    answer   = str(example.get('answer', '')).strip()
    context  = example.get('context', '').strip()

    if not question or not answer:
        return None

    if context:
        user_content = f'Financial context:\n{context}\n\nQuestion: {question}'
    else:
        user_content = f'Question: {question}'

    text = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{user_content}<|end|>\n'
        f'<|assistant|>\n{answer}<|end|>'
    )
    return {'text': text, 'question': question, 'answer': answer}

ds = load_dataset('virattt/financial-qa-10K')

formatted, skipped = [], 0
for ex in ds['train']:
    r = format_phi3(ex)
    if r:
        formatted.append(r)
    else:
        skipped += 1

print(f'Formatados: {len(formatted)} | Ignorados: {skipped}')

random.seed(42)
random.shuffle(formatted)
eval_size  = int(len(formatted) * 0.15)
train_data = formatted[eval_size:]
eval_data  = formatted[:eval_size]

train_ds = Dataset.from_list(train_data)
eval_ds  = Dataset.from_list(eval_data)

print(f'Treino: {len(train_ds)} | Eval: {len(eval_ds)}')
print('\n--- Exemplo ---')
print(train_ds[0]['text'][:500])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/419 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Formatados: 6997 | Ignorados: 3
Treino: 5948 | Eval: 1049

--- Exemplo ---
<|system|>
You are an expert assistant in corporate finance. Answer questions about balance sheets, income statements, cash flow and accounting terms clearly and accurately.<|end|>
<|user|>
Financial context:
In 2023, under pre-approved share repurchase programs, The Hershey Company repurchased shares valued at $27.4 million.

Question: What is the value of shares repurchased under the pre-approved program as stated in The Hershey Company's 2023 Form 10-K, for the year 2023?<|end|>
<|assistant|>


## 4. Carregar modelo base (baseline)

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Carregando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN, trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Carregando modelo em 4-bit...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    attn_implementation='eager',
)

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f'Modelo carregado! VRAM usada: {vram_used:.2f} GB')

Carregando tokenizer...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Modelo carregado! VRAM usada: 2.11 GB


## 5. Avaliar baseline

In [8]:
TEST_QUESTIONS = [
    'What is EBITDA and how is it calculated?',
    'What is the difference between gross profit and net profit?',
    'What does a current ratio below 1 indicate?',
    'How do you analyze the free cash flow of a company?',
    'What is the difference between accounts payable and accounts receivable?',
]

def generate_answer(model, tokenizer, question, max_new_tokens=200):
    prompt = (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\nQuestion: {question}<|end|>\n'
        f'<|assistant|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('=== BASELINE - Phi-3-mini SEM fine-tuning ===')
baseline_answers = {}
for q in TEST_QUESTIONS:
    answer = generate_answer(base_model, tokenizer, q)
    baseline_answers[q] = answer
    print(f'\nQ: {q}')
    print(f'A: {answer[:300]}')
    print('-' * 50)

=== BASELINE - Phi-3-mini SEM fine-tuning ===

Q: What is EBITDA and how is it calculated?
A: EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is a measure used to evaluate a company'limpact of its operational performance without the effects of financing and accounting decisions. EBITDA is calculated by taking a company's net income and adding back intere
--------------------------------------------------

Q: What is the difference between gross profit and net profit?
A: Gross profit and net profit are two key financial metrics used to assess a company'ieves.


Gross profit is the profit a company makes after deducting the costs directly associated with the production of the goods it sells, which is known as the cost of goods sold (COGS). It is calculated by subtrac
--------------------------------------------------

Q: What does a current ratio below 1 indicate?
A: A current ratio below 1 indicates that a company may not have enough liquid assets t

## 6. Aplicar QLoRA e treinar

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

base_model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,      # era 2
    per_device_eval_batch_size=4,       # era 2
    gradient_accumulation_steps=4,      # era 8 — effective batch continua 16
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,                    # trocou warmup_ratio
    weight_decay=0.01,
    optim='paged_adamw_8bit',
    bf16=True,
    max_length=256,                     # era 512 — maior impacto na velocidade
    dataset_text_field='text',
    packing=True,                       # era False — empacota sequências, muito mais rápido
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=50,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

train_ds_small = train_ds.select(range(1000))
eval_ds_small  = eval_ds.select(range(200))

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_small,
    eval_dataset=eval_ds_small,
    args=training_args,
)

print('Iniciando treino...')
trainer.train()

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Iniciando treino...


Epoch,Training Loss,Validation Loss
1,No log,0.850359
2,1.367151,0.810827
3,0.849450,0.806818


TrainOutput(global_step=132, training_loss=1.0354951511729846, metrics={'train_runtime': 7921.4606, 'train_samples_per_second': 0.264, 'train_steps_per_second': 0.017, 'total_flos': 1.04353781349888e+16, 'train_loss': 1.0354951511729846})

In [10]:
model.save_pretrained('/content/finance-phi3-qlora')

## 7. Salvar modelo

In [11]:
SAVE_DIR = '/content/finance-phi3-qlora'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Modelo salvo em {SAVE_DIR}')
!ls -lh /content/finance-phi3-qlora/

Modelo salvo em /content/finance-phi3-qlora
total 38M
-rw-r--r-- 1 root root 1.1K Mar 30 18:01 adapter_config.json
-rw-r--r-- 1 root root  35M Mar 30 18:01 adapter_model.safetensors
-rw-r--r-- 1 root root  407 Mar 30 18:01 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Mar 30 18:01 README.md
-rw-r--r-- 1 root root  381 Mar 30 18:01 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Mar 30 18:01 tokenizer.json


## 8. Avaliar modelo fine-tunado

In [12]:
print('=== FINE-TUNED - Phi-3-mini APOS fine-tuning ===')
finetuned_answers = {}
for q in TEST_QUESTIONS:
    answer = generate_answer(model, tokenizer, q)
    finetuned_answers[q] = answer
    print(f'\nQ: {q}')
    print(f'A: {answer[:300]}')
    print('-' * 50)

=== FINE-TUNED - Phi-3-mini APOS fine-tuning ===

Q: What is EBITDA and how is it calculated?
A: EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is calculated by adding back interest, taxes, depreciation, and amortization to a company'cial earnings.
--------------------------------------------------

Q: What is the difference between gross profit and net profit?
A: Gross profit is the difference between revenue and the cost of goods sold, while net profit is the difference between revenue and all expenses, including taxes and interest.
--------------------------------------------------

Q: What does a current ratio below 1 indicate?
A: A current ratio below 1 indicates that a company'cial financial health.
--------------------------------------------------

Q: How do you analyze the free cash flow of a company?
A: To analyze the free cash flow of a company, you need to understand the context of the company's operations, financial health, and indust

## 9. Comparacao ROUGE + tabela visual

In [13]:
import evaluate
import pandas as pd
from IPython.display import display, HTML

rouge = evaluate.load('rouge')

reference_answers = {
    'What is EBITDA and how is it calculated?':
        'EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It is calculated by adding back interest, taxes, depreciation and amortization to net income.',
    'What is the difference between gross profit and net profit?':
        'Gross profit is revenue minus cost of goods sold. Net profit is gross profit minus all operating expenses, interest, and taxes.',
    'What does a current ratio below 1 indicate?':
        'A current ratio below 1 indicates the company has more current liabilities than current assets, suggesting potential liquidity problems.',
    'How do you analyze the free cash flow of a company?':
        'Free cash flow is calculated as operating cash flow minus capital expenditures. Positive and growing FCF indicates financial health.',
    'What is the difference between accounts payable and accounts receivable?':
        'Accounts payable is money owed to suppliers. Accounts receivable is money owed by customers to the company.',
}

rows = []
for q in TEST_QUESTIONS:
    ref = reference_answers[q]
    base = baseline_answers[q]
    ft = finetuned_answers[q]
    r_base = rouge.compute(predictions=[base], references=[ref])
    r_ft = rouge.compute(predictions=[ft], references=[ref])
    rows.append({
        'Pergunta': q[:55] + '...',
        'ROUGE-L Base': round(r_base['rougeL'], 3),
        'ROUGE-L Fine-tuned': round(r_ft['rougeL'], 3),
        'Delta': round(r_ft['rougeL'] - r_base['rougeL'], 3),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f'\nMedia Base: {df["ROUGE-L Base"].mean():.3f}')
print(f'Media Fine-tuned: {df["ROUGE-L Fine-tuned"].mean():.3f}')
print(f'Melhoria media: +{df["Delta"].mean():.3f}')

html_rows = ''
for q in TEST_QUESTIONS:
    base = baseline_answers[q][:350]
    ft = finetuned_answers[q][:350]
    html_rows += f'<tr style="background:#f9f9f9"><td colspan="2" style="padding:10px;font-weight:bold">{q}</td></tr>'
    html_rows += f'<tr><td style="padding:10px;width:50%;border-right:1px solid #ddd;vertical-align:top"><b style="color:#c0392b">Base</b><br><br>{base}</td><td style="padding:10px;vertical-align:top"><b style="color:#27ae60">Fine-tuned</b><br><br>{ft}</td></tr>'
    html_rows += '<tr><td colspan="2" style="border-bottom:2px solid #eee"></td></tr>'

html = f'<table style="width:100%;border-collapse:collapse;font-size:13px"><thead><tr style="background:#2c3e50;color:white"><th style="padding:12px">Phi-3-mini Base</th><th style="padding:12px">Phi-3-mini Fine-tuned (FinQA)</th></tr></thead><tbody>{html_rows}</tbody></table>'
display(HTML(html))

                                                  Pergunta  ROUGE-L Base  ROUGE-L Fine-tuned  Delta
               What is EBITDA and how is it calculated?...         0.348               0.880  0.532
What is the difference between gross profit and net pro...         0.234               0.571  0.338
            What does a current ratio below 1 indicate?...         0.246               0.452  0.206
    How do you analyze the free cash flow of a company?...         0.119               0.185  0.065
What is the difference between accounts payable and acc...         0.142               0.407  0.265

Media Base: 0.218
Media Fine-tuned: 0.499
Melhoria media: +0.281


## 10. Salvar CSV e fazer download

In [14]:
df.to_csv('/content/results_comparison.csv', index=False)
print('Salvo em /content/results_comparison.csv')

try:
    from google.colab import files
    files.download('/content/results_comparison.csv')
except Exception:
    print('Download manual: clique no arquivo no painel Files do Colab')

Salvo em /content/results_comparison.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. (Opcional) Upload para o Hugging Face Hub

In [18]:
HF_USERNAME = 'Shinigami4242557'
REPO_NAME = 'phi3-mini-finance-qlora'
model.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}', token=HF_TOKEN)
tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}', token=HF_TOKEN)
print(f'Publicado em: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|3         | 1.20MB / 35.7MB            

README.md: 0.00B [00:00, ?B/s]

Publicado em: https://huggingface.co/Shinigami4242557/phi3-mini-finance-qlora


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    '/content/finance-phi3-qlora',
    '/content/drive/MyDrive/finance-phi3-qlora'
)
print("✓ Salvo no Drive!")

Mounted at /content/drive
✓ Salvo no Drive!


In [2]:
# ── Instalar dependências ─────────────────────────────────────────────────────
!pip install -q gradio peft bitsandbytes accelerate

# ── Montar Drive e carregar adaptadores ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_PATH = "/content/drive/MyDrive/finance-phi3-qlora"

# ── Imports ───────────────────────────────────────────────────────────────────
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL   = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 300
TEMPERATURE    = 0.7

# ── Carregar modelo ───────────────────────────────────────────────────────────
print("Loading base model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,  # 👈
    attn_implementation="eager",  # 👈
)

print("Applying LoRA adapters...")
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=False)
print("✓ Model ready!")

# ── Inferência ────────────────────────────────────────────────────────────────
def respond(question: str) -> str:
    if not question.strip():
        return "Please enter a question."

    prompt = (
        "<|system|>\n"
        "You are a concise financial expert assistant. "
        "Answer questions about corporate finance, accounting, and financial analysis accurately and directly.\n"
        "<|end|>\n"
        "<|user|>\n"
        f"{question.strip()}\n"
        "<|end|>\n"
        "<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ── Interface Gradio ──────────────────────────────────────────────────────────
EXAMPLES = [
    "What is EBITDA and how is it calculated?",
    "What is the difference between gross profit and net profit?",
    "What does a current ratio below 1 indicate?",
    "How do you analyze the free cash flow of a company?",
    "What is the difference between accounts payable and accounts receivable?",
]

with gr.Blocks(title="Finance Phi-3 QLoRA") as demo:
    gr.Markdown("""
    # Finance Phi-3 QLoRA
    **Phi-3-mini-4k-instruct** fine-tuned on financial Q&A via QLoRA
    `0.23% parameters trained · ROUGE-L +129%`
    """)

    question = gr.Textbox(label="Question", placeholder="Ask a financial question...", lines=2)
    btn      = gr.Button("Ask", variant="primary")
    answer   = gr.Textbox(label="Answer", lines=6, interactive=False)

    gr.Examples(examples=EXAMPLES, inputs=question)

    btn.click(fn=respond, inputs=question, outputs=answer)
    question.submit(fn=respond, inputs=question, outputs=answer)

demo.launch(share=True)  # share=True gera link público

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading base model...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Applying LoRA adapters...
✓ Model ready!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9bc557e6195e6911ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
